# MawinguOps — ML Model Training

This notebook trains the **planting recommendation** Random Forest used by
MawinguOps, a USSD-based early-warning and planting-advisory system for
smallholder maize farmers in Machakos County, Kenya.

**What it does**
1. Pulls the CHIRPS V3 daily rainfall archive (1998–2025, the available range
   for this location) for Machakos from Google Earth Engine.
2. Computes the day-of-year climatological baseline.
3. Engineers 10 agronomic features per growing season (MAM & OND).
4. Labels each season with FAO maize water-requirement thresholds.
5. Trains a `RandomForestClassifier` and evaluates it.

**Outputs** (download at the end and place in the repo):
- `planting_model.pkl`  → `ml/models/`
- `label_encoder.pkl`   → `ml/models/`
- `feature_importance.json` → `ml/models/`
- `chirps_machakos.csv`, `baseline_machakos.csv` → `pipeline/data/`

In [ ]:
!pip install earthengine-api scikit-learn joblib pandas numpy matplotlib seaborn psycopg2-binary python-dotenv

In [ ]:
import ee

# Replace with your own Google Cloud project ID that has the Earth Engine API enabled.
EE_PROJECT = 'your-google-cloud-project-id'  # <-- CHANGE ME

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print('Earth Engine initialised for project:', EE_PROJECT)

In [ ]:
MACHAKOS_LAT = -1.5177
MACHAKOS_LON = 37.2634
LOCATION = 'machakos'
# CHIRPS V3 DAILY_SAT begins in 1998 for this location, so the archive starts there.
START_DATE = '1998-01-01'
END_DATE = '2025-12-31'
CHIRPS_COLLECTION = 'UCSB-CHC/CHIRPS/V3/DAILY_SAT'

print(f'Date range: {START_DATE} to {END_DATE}')
print(f'Location: {MACHAKOS_LAT}, {MACHAKOS_LON}')

## 1. Fetch CHIRPS daily rainfall from Earth Engine


In [ ]:
import pandas as pd
import numpy as np

point = ee.Geometry.Point([MACHAKOS_LON, MACHAKOS_LAT])

collection = (ee.ImageCollection(CHIRPS_COLLECTION)
              .filterDate(START_DATE, END_DATE)
              .select('precipitation'))

def to_feature(image):
    value = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=5566,  # CHIRPS native ~0.05 deg
    ).get('precipitation')
    return ee.Feature(None, {
        'date': image.date().format('YYYY-MM-dd'),
        'rainfall_mm': value,
    })

# CHIRPS V3 daily is large; fetch year-by-year to stay under the getInfo limit.
frames = []
for year in range(1998, 2026):
    yr_col = collection.filterDate(f'{year}-01-01', f'{year}-12-31')
    feats = yr_col.map(to_feature).getInfo()['features']
    recs = [f['properties'] for f in feats if f['properties'].get('rainfall_mm') is not None]
    if recs:
        frames.append(pd.DataFrame(recs))
    print(f'  {year}: {len(recs)} days')

chirps = pd.concat(frames, ignore_index=True)
chirps['date'] = pd.to_datetime(chirps['date'])
chirps['rainfall_mm'] = chirps['rainfall_mm'].astype(float)
chirps = chirps.sort_values('date').reset_index(drop=True)

print('Shape:', chirps.shape)
chirps.head()

In [ ]:
chirps.to_csv('chirps_machakos.csv', index=False)
print('Saved chirps_machakos.csv')

## 2. Compute the historical day-of-year baseline (1998–2020)

In [ ]:
baseline_src = chirps[chirps['date'].dt.year <= 2020].copy()
baseline_src['day_of_year'] = baseline_src['date'].dt.dayofyear
baseline = (baseline_src.groupby('day_of_year')['rainfall_mm']
            .mean().rename('mean_rainfall_mm').reset_index())
print('Baseline days:', baseline.shape[0])
baseline.to_csv('baseline_machakos.csv', index=False)
baseline.head()

## 3. Engineer features

The feature-engineering logic mirrors `ml/features.py` (copied inline so the
notebook runs standalone). One row per (year, season).


In [ ]:
DRY_DAY_THRESHOLD_MM = 1.0
ONSET_WINDOW_MM = 20.0

FEATURE_NAMES = [
    'onset_week', 'cumulative_rainfall_30d', 'dry_spell_max',
    'rainfall_variability', 'anomaly_pct', 'forecast_7d_total',
    'forecast_14d_total', 'forecast_dry_spell', 'season', 'week_of_season',
]

# (season_code, start_month, start_day, end_month, end_day)
SEASONS = [(0, 3, 1, 5, 31), (1, 10, 1, 12, 31)]  # MAM, OND

def longest_dry_spell(values):
    longest = current = 0
    for v in values:
        if v < DRY_DAY_THRESHOLD_MM:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return int(longest)

def compute_onset_week(season_df):
    values = season_df['rainfall_mm'].to_numpy()
    for week in range(len(values) // 7):
        if values[week*7:(week+1)*7].sum() >= ONSET_WINDOW_MM:
            return week + 1
    return 0

doy_baseline = baseline.set_index('day_of_year')['mean_rainfall_mm']

rows = []
for year in sorted(chirps['date'].dt.year.unique()):
    for (season_code, sm, sd, em, ed) in SEASONS:
        start = pd.Timestamp(year=year, month=sm, day=sd)
        end = pd.Timestamp(year=year, month=em, day=ed)
        sdf = chirps[(chirps['date'] >= start) & (chirps['date'] <= end)]
        if len(sdf) < 30:
            continue
        values = sdf['rainfall_mm'].to_numpy()
        first_30 = values[:30]
        cumulative_rainfall_30d = float(first_30.sum())
        dry_spell_max = longest_dry_spell(first_30)
        rainfall_variability = float(np.std(first_30))
        season_doys = sdf['date'].dt.dayofyear.to_numpy()[:30]
        baseline_sum = float(doy_baseline.reindex(season_doys).fillna(0).sum())
        anomaly_pct = (cumulative_rainfall_30d / baseline_sum * 100) if baseline_sum > 0 else 0.0
        next_14 = values[30:44]
        forecast_7d_total = float(next_14[:7].sum())
        forecast_14d_total = float(next_14[:14].sum())
        forecast_dry_spell = longest_dry_spell(next_14)
        rows.append({
            'year': int(year),
            'season_label': 'MAM' if season_code == 0 else 'OND',
            'onset_week': compute_onset_week(sdf),
            'cumulative_rainfall_30d': round(cumulative_rainfall_30d, 2),
            'dry_spell_max': dry_spell_max,
            'rainfall_variability': round(rainfall_variability, 2),
            'anomaly_pct': round(anomaly_pct, 2),
            'forecast_7d_total': round(forecast_7d_total, 2),
            'forecast_14d_total': round(forecast_14d_total, 2),
            'forecast_dry_spell': forecast_dry_spell,
            'season': int(season_code),
            'week_of_season': 4,
        })

features_df = pd.DataFrame(rows)
print('Feature rows:', features_df.shape)
features_df.head()

## 4. Generate labels (FAO maize thresholds)

Labels are derived **programmatically from agronomic thresholds, not from
actual yield data** (FAO Irrigation and Drainage Paper 56).


In [ ]:
# FAO-derived thresholds (mm and days).
GERMINATION_MIN_MM = 25.0
GERMINATION_BORDERLINE_MM = 30.0
GERMINATION_MAX_DRY_DAYS = 7
EARLY_2WEEK_MIN_MM = 25.0

# Hard-failure thresholds (clearly unsuitable conditions).
HARD_FAIL_MIN_MM = 15.0
HARD_FAIL_DRY_DAYS = 12

def label_row(row):
    germination_mm = row['cumulative_rainfall_30d']
    dry_spell_max = row['dry_spell_max']
    forecast_14d = row['forecast_14d_total']
    forecast_dry_spell = row['forecast_dry_spell']

    # DO_NOT_PLANT: clearly unsuitable (almost no rain, or a severe dry spell).
    if (germination_mm < HARD_FAIL_MIN_MM
            or dry_spell_max > HARD_FAIL_DRY_DAYS
            or forecast_dry_spell > HARD_FAIL_DRY_DAYS):
        return 'DO_NOT_PLANT'

    # WAIT: marginal, reassess next week. A >7 day current dry spell lands here
    # (not DO_NOT_PLANT) because short gaps are normal in semi-arid Machakos.
    if (germination_mm < GERMINATION_MIN_MM
            or dry_spell_max > GERMINATION_MAX_DRY_DAYS
            or 6 <= forecast_dry_spell <= HARD_FAIL_DRY_DAYS
            or germination_mm < GERMINATION_BORDERLINE_MM
            or forecast_14d < EARLY_2WEEK_MIN_MM):
        return 'WAIT'

    # PLANT_NOW: needs met and a wet, stable 2-week outlook.
    if forecast_14d >= EARLY_2WEEK_MIN_MM and forecast_dry_spell < 6:
        return 'PLANT_NOW'

    return 'WAIT'

labels = features_df.apply(label_row, axis=1).rename('label')
print(labels.value_counts())

## 5. Train the Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib

X = features_df[FEATURE_NAMES]
encoder = LabelEncoder()
y = encoder.fit_transform(labels)

stratify = y if len(set(y)) > 1 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Pass explicit label indices so every class is reported even when a small,
# imbalanced class is absent from the test split.
label_indices = list(range(len(encoder.classes_)))
print(classification_report(
    y_test, y_pred, labels=label_indices,
    target_names=encoder.classes_, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred, labels=label_indices))

## 6. Evaluate and visualise


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))

importances = pd.Series(model.feature_importances_, index=FEATURE_NAMES).sort_values()
plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='#2563eb')
plt.title('Feature importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

# Use explicit labels so the matrix is always 3x3 and aligns with the tick labels.
cm = confusion_matrix(y_test, y_pred, labels=label_indices)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title('Confusion matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 7. Save the model artifacts


In [ ]:
import json

joblib.dump(model, 'planting_model.pkl')
joblib.dump(encoder, 'label_encoder.pkl')

feature_importance = dict(zip(FEATURE_NAMES, model.feature_importances_.tolist()))
with open('feature_importance.json', 'w') as f:
    json.dump(feature_importance, f, indent=2)

print('Model saved. Download planting_model.pkl and label_encoder.pkl and place them in ml/models/')

## 8. Download files (Colab)


In [ ]:
from google.colab import files
files.download('planting_model.pkl')
files.download('label_encoder.pkl')
files.download('feature_importance.json')
files.download('chirps_machakos.csv')
files.download('baseline_machakos.csv')

## Next steps

1. Place `planting_model.pkl`, `label_encoder.pkl` and
   `feature_importance.json` in **`ml/models/`**.
2. Place `chirps_machakos.csv` and `baseline_machakos.csv` in
   **`pipeline/data/`**.
3. The weekly cron job (`pipeline/run_pipeline.sh`) loads
   `ml/models/planting_model.pkl` in `run_model.py`, engineers the same 10
   features from the live database, runs `predict()` / `predict_proba()`, and
   stores the recommendation in `planting_recommendations`.
4. `generate_advisory.py` then turns that recommendation into a plain-language
   Swahili/English advisory via Llama (hosted on Groq), which the USSD handler
   reads back to farmers.

The USSD handler never runs ML inference live — it only reads pre-computed rows.